In [1]:
import h5py
import numpy as np
import os

path1 = r"C:\Users\gabri\Desktop\stage_sommeil\data_serveur"
path2 = r"C:\Users\gabri\Desktop\stage_sommeil\algo\sleep_fm\sleepfm-clinical\notebooks\train_data_more\train_emb_test"

# Fichiers HDF5 communs aux deux dossiers
files1 = {f for f in os.listdir(path1) if f.endswith(".hdf5")}
files2 = {f for f in os.listdir(path2) if f.endswith(".hdf5")}
common = sorted(files1 & files2)

print(f"Fichiers communs : {len(common)}")
print("-" * 60)

results = {}

for fname in common:
    subject_id = fname.replace(".hdf5", "")
    p1 = os.path.join(path1, fname)
    p2 = os.path.join(path2, fname)

    with h5py.File(p1, "r") as f1, h5py.File(p2, "r") as f2:
        modalities = list(f1.keys())
        subject_stats = {}

        for mod in modalities:
            e1 = f1[mod][:]
            e2 = f2[mod][:]

            if e1.shape != e2.shape:
                print(f"[{subject_id}] {mod} : shapes différentes {e1.shape} vs {e2.shape}")
                continue

            diff = e1 - e2
            abs_diff = np.abs(diff)

            subject_stats[mod] = {
                "mean_abs_diff"   : float(abs_diff.mean()),
                "max_abs_diff"    : float(abs_diff.max()),
                "std_diff"        : float(diff.std()),
                "cosine_sim_mean" : float(
                    np.mean([
                        np.dot(e1[i], e2[i]) /
                        (np.linalg.norm(e1[i]) * np.linalg.norm(e2[i]) + 1e-8)
                        for i in range(len(e1))
                    ])
                ),
            }

            print(
                f"[{subject_id}] {mod:10s} | "
                f"mean |diff|={subject_stats[mod]['mean_abs_diff']:.6f}  "
                f"max |diff|={subject_stats[mod]['max_abs_diff']:.6f}  "
                f"cos_sim={subject_stats[mod]['cosine_sim_mean']:.6f}"
            )

        results[subject_id] = subject_stats

print("-" * 60)
print("Résumé global par modalité :")
all_mods = {mod for s in results.values() for mod in s}
for mod in sorted(all_mods):
    vals = [results[s][mod]["mean_abs_diff"] for s in results if mod in results[s]]
    print(f"  {mod:10s} | mean |diff| moyen sur tous les sujets : {np.mean(vals):.6f}")

Fichiers communs : 14
------------------------------------------------------------
[C1_001_PSG1] BAS        | mean |diff|=0.000001  max |diff|=0.000006  cos_sim=1.000000
[C1_001_PSG1] EKG        | mean |diff|=0.000290  max |diff|=0.003238  cos_sim=1.000000
[C1_001_PSG1] EMG        | mean |diff|=0.000001  max |diff|=0.000006  cos_sim=1.000000
[C1_001_PSG1] RESP       | mean |diff|=0.000001  max |diff|=0.000006  cos_sim=1.000000
[C1_002_PSG1] BAS        | mean |diff|=0.000001  max |diff|=0.000006  cos_sim=1.000000
[C1_002_PSG1] EKG        | mean |diff|=0.000263  max |diff|=0.003187  cos_sim=1.000000
[C1_002_PSG1] EMG        | mean |diff|=0.000001  max |diff|=0.000006  cos_sim=1.000000
[C1_002_PSG1] RESP       | mean |diff|=0.000001  max |diff|=0.000006  cos_sim=1.000000
[C1_003_PSG1] BAS        | mean |diff|=0.000001  max |diff|=0.000005  cos_sim=1.000000
[C1_003_PSG1] EKG        | mean |diff|=0.000199  max |diff|=0.002509  cos_sim=1.000000
[C1_003_PSG1] EMG        | mean |diff|=0.000001